# Ken Kahn Research Posts — Incremental Table Builder

**Run All** updates the table of research posts from the master Google Doc, without redoing old work:
`ken_kahn_posts.json` (the data) and `ken_kahn_posts.html` (the table). The table has a toggle between
**All posts** and **Web apps that use AI**; the second view has its own address, `ken_kahn_posts.html#ai-apps`,
and the old `apps_with_AI.html` now forwards there.

How it works:

1. Earlier results are loaded — from files in the current folder if present, otherwise from the GitHub repo.
2. The master Google Doc is read to get its current list of posts.
3. Posts seen before are reused as-is. Only **new** posts are fetched, have their links checked, get an AI
   description, have their app links sorted by AI into apps and docs (grouping copies and versions of one app,
   which gives the *unique apps* count), and get an AI verdict on whether they belong on the
   *web apps that use AI* list.
   The verdict is made by showing the AI all the earlier in/out decisions, so it follows the same line;
   each verdict is printed with its reason, and `AI_APPS_ADD` / `AI_APPS_REMOVE` overrule it.
4. Anything that failed (a description or a verdict) is retried on the next run.
5. Files are saved (and downloaded in Colab) only if they changed.

Progress is saved after every post, so if a run is interrupted, **Run All** again picks up where it stopped.
Run from a local copy of the repo folder (e.g. by Claude Code), the files there are updated in place.

**API key:** OpenAI by default. In Colab, add a secret named `OPENAI_API_KEY`
(key icon in the left sidebar) — or `GEMINI_API_KEY` if `PROVIDER = 'gemini'`.
If there's no secret you'll be asked for the key, but only when there's something new to do.

## ⚙️ Configuration

In [ ]:
MASTER_URL = "https://docs.google.com/document/d/e/2PACX-1vS9UYVtIhtzPut25FyiLX1EpYkN8U0dDjEAzg44z6bf21POmu0zQ7WH1WJQFBwxO0hZIxJUUWkW1Zm-/pub"

OUTPUT = 'ken_kahn_posts'         # -> ken_kahn_posts.json + ken_kahn_posts.html
TITLE = None                      # page title; None = the master doc's own title
AI_APPS_TITLE = 'Recent research by Ken Kahn: web apps that use AI'   # page title in the AI-apps view
# The old address of the separate AI-apps table; it now forwards to the AI-apps view. None = don't write it.
AI_APPS_REDIRECT = 'apps_with_AI.html'

# Overrule the AI's verdict on a post: put (part of) its title, or its doc ID, here.
AI_APPS_ADD = []
AI_APPS_REMOVE = []

# Overrule how app links were sorted. Rename or merge apps: {'name the AI gave': 'name to use'}.
APP_RENAMES = {}
# Links (or parts of their addresses) that are apps even though the AI called them docs, and the reverse.
FORCE_APP_LINKS = []
FORCE_DOC_LINKS = []
CONSOLIDATE_APP_NAMES = False     # True = also ask the AI to merge app names that drifted apart across posts

# Earlier results: a file in the current folder wins; otherwise it's fetched from here.
GITHUB_RAW = "https://raw.githubusercontent.com/ToonTalk/AI/main/misc/turning%20posts%20into%20a%20table/"

PROVIDER = 'openai'               # 'openai' or 'gemini'
OPENAI_MODEL = 'gpt-5-mini'
GEMINI_MODEL = 'gemini-2.5-flash'

# Redo the links and description of specific posts even though they're already known: (part of) a title or a doc ID.
REFRESH = []
REDO_ALL = False                  # True = redo every post's links and description (not its verdict)
DRY_RUN = False                   # True = just report what would be done; no fetching posts, no AI calls

print("✅ Configuration loaded")

## 🔧 Helper functions

In [ ]:
import re
import os
import json
import time
import urllib.request
import urllib.parse
import urllib.error
from html.parser import HTMLParser
import html as html_lib

# Global tracker for broken links
broken_links = []


class LinkExtractor(HTMLParser):
    def __init__(self):
        super().__init__()
        self.links = []
        self.current_link = None
        self.current_text = []

    def handle_starttag(self, tag, attrs):
        if tag == 'a':
            href = dict(attrs).get('href', '')
            if href:
                self.current_link = href
                self.current_text = []

    def handle_endtag(self, tag):
        if tag == 'a' and self.current_link:
            self.links.append({'url': self.current_link, 'text': ''.join(self.current_text).strip()})
            self.current_link = None

    def handle_data(self, data):
        if self.current_link is not None:
            self.current_text.append(data)


class TitleExtractor(HTMLParser):
    def __init__(self):
        super().__init__()
        self.title = None
        self.in_title = False

    def handle_starttag(self, tag, attrs):
        if tag == 'title':
            self.in_title = True

    def handle_endtag(self, tag):
        if tag == 'title':
            self.in_title = False

    def handle_data(self, data):
        if self.in_title:
            self.title = data.strip()


class TextExtractor(HTMLParser):
    def __init__(self):
        super().__init__()
        self.text = []
        self.skip_tags = {'script', 'style', 'head'}
        self.current_tag = None

    def handle_starttag(self, tag, attrs):
        self.current_tag = tag

    def handle_endtag(self, tag):
        self.current_tag = None

    def handle_data(self, data):
        if self.current_tag not in self.skip_tags:
            self.text.append(data)

    def get_text(self):
        return ' '.join(self.text)


def extract_markdown_links(text):
    links = []
    for match in re.finditer(r'\[([^\]]+)\]\(([^\)]+)\)', text):
        links.append({'url': match.group(2), 'text': match.group(1)})
    for match in re.finditer(r'https?://[^\s<>"\')]+', text):
        url = match.group(0)
        if not any(link['url'] == url for link in links):
            links.append({'url': url, 'text': ''})
    return links


def extract_all_links(html_content):
    parser = LinkExtractor()
    parser.feed(html_content)
    all_links = parser.links + extract_markdown_links(html_content)
    seen = set()
    unique = []
    for link in all_links:
        if link['url'] not in seen:
            seen.add(link['url'])
            unique.append(link)
    return unique


def extract_youtube_id(url):
    for pattern in [r'youtu\.be/([a-zA-Z0-9_-]{11})', r'youtube\.com/watch\?v=([a-zA-Z0-9_-]{11})']:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    return None


def fetch_youtube_title(video_id, link_text=''):
    video_url = f"https://www.youtube.com/watch?v={video_id}"
    try:
        req = urllib.request.Request(
            f"https://www.youtube.com/oembed?url={video_url}&format=json",
            headers={'User-Agent': 'Mozilla/5.0'}
        )
        with urllib.request.urlopen(req, timeout=10) as response:
            return json.loads(response.read().decode('utf-8')).get('title', f'YouTube Video {video_id}')
    except Exception as e:
        # Skip 403 Forbidden errors
        if hasattr(e, 'code') and e.code == 403:
            return f'YouTube Video {video_id}'
        error_msg = str(e)
        if hasattr(e, 'code'):
            error_msg = f"HTTP {e.code}" + (" (Video may be private or deleted)" if e.code == 404 else "")
        broken_links.append({'url': video_url, 'error': error_msg, 'type': 'video',
                             'context': link_text or 'No link text available'})
        return f'YouTube Video {video_id}'


def is_youtube_url(url):
    return ('youtube.com' in url.lower() or 'youtu.be' in url.lower()) and extract_youtube_id(url)


def is_auth_protected_url(url):
    return 'claude.ai/share' in url.lower() or 'gemini' in url.lower()


def unwrap_google_redirect(url):
    if 'google.com/url?' in url:
        params = urllib.parse.parse_qs(urllib.parse.urlparse(url).query)
        if 'q' in params:
            return params['q'][0]
    return url


def is_google_doc_link(url):
    return 'docs.google.com/document' in url.lower()


def doc_key(url):
    """A stable key for a Google Doc, so '/edit?usp=sharing' and '/edit' variants match."""
    match = re.search(r'/document/d/(e/)?([a-zA-Z0-9_-]+)', url)
    return match.group(2) if match else url.strip()


def is_artifact_link(text, url):
    text_lower = text.lower()
    if any(x in text_lower and len(text_lower) < 30 for x in ['report abuse', 'home page', 'ken kahn']):
        return True
    return any(d in url.lower() for d in ['linkedin.com', 'facebook.com', 'threads.net', 'bsky.app'])


def fetch_page_title(url, default='', link_text=''):
    if is_auth_protected_url(url):
        return default
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=10) as response:
            parser = TitleExtractor()
            parser.feed(response.read(10000).decode('utf-8', errors='ignore'))
            if parser.title:
                return re.sub(r'\s*[\|\-]\s*(Claude|ChatGPT|Gemini).*$', '', parser.title, flags=re.I).strip() or default
    except Exception as e:
        # Skip 403 Forbidden errors (authentication required)
        if hasattr(e, 'code') and e.code == 403:
            return default
        error_msg = str(e)
        if hasattr(e, 'code'):
            error_msg = f"HTTP {e.code}"
        elif 'timed out' in error_msg.lower():
            error_msg = "Timeout"
        elif 'urlopen error' in error_msg.lower():
            error_msg = "Connection failed"
        broken_links.append({'url': url, 'error': error_msg, 'type': 'page',
                             'context': link_text or 'No link text available'})
    return default


def toontalk_app_title(url):
    """A readable default title for a toontalk.github.io app: its file name, or its folder's name."""
    path = urllib.parse.unquote(urllib.parse.urlparse(url).path).rstrip('/')
    name = path.rsplit('/', 1)[-1]
    is_jsx = name.lower().endswith('.jsx')
    name = re.sub(r'\.(html?|jsx)$', '', name, flags=re.I)
    name = re.sub(r'\s+', ' ', re.sub(r'(?<=\w)[-_](?=\w)', ' ', name)).strip() or 'ToonTalk'
    return name + ' (JSX)' if is_jsx else name


def classify_link(url, text=''):
    url = unwrap_google_redirect(url)
    if any(p in text.lower() for p in ['early version', 'first version']):
        return 'skip', ''
    if is_youtube_url(url):
        return 'video', f'YouTube Video {extract_youtube_id(url)}'

    # ToonTalk links: a web page, a .jsx app, or a folder (which serves its index.html) is an app;
    # the site's home page and other files are not
    if 'toontalk.github.io' in url.lower():
        try:
            path = urllib.parse.urlparse(url).path.lower()
        except ValueError:   # a malformed address
            return 'other', 'Hosted Web Page'
        if path not in ('', '/') and (path.endswith(('.html', '.htm', '.jsx')) or path.endswith('/')):
            return 'app', toontalk_app_title(url)
        return 'other', 'Hosted Web Page'

    for pattern, title in [(r'chatgpt\.com/canvas', 'ChatGPT Canvas'), (r'claude\.ai/(.*/)?artifacts?/', 'Claude Artifact')]:
        if re.search(pattern, url.lower()):
            return 'app', title

    for pattern, title in [(r'chatgpt\.com/share', 'ChatGPT Chat'), (r'chat\.openai\.com/share', 'ChatGPT Chat'), (r'claude\.ai/share', 'Claude Chat')]:
        if re.search(pattern, url.lower()):
            return 'chat', title

    if 'gemini' in url.lower():
        return 'app' if any(k in text.lower() for k in ['app', 'game', 'tool']) else 'chat', 'Gemini'
    return 'other', text


def fetch_google_doc_html(url):
    if '/edit' in url:
        match = re.search(r'/document/d/([a-zA-Z0-9_-]+)', url)
        url = f'https://docs.google.com/document/d/{match.group(1)}/export?format=html' if match else url
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=60) as response:   # long posts with images can be slow to export
        return response.read().decode('utf-8')


def infer_chatbots(apps, chats, videos, title):
    bots = set()
    if title:
        t = title.lower()
        if 'chatgpt' in t or 'gpt' in t:
            bots.add('ChatGPT')
        if 'claude' in t:
            bots.add('Claude')
        if 'gemini' in t:
            bots.add('Gemini')
    for item in apps + chats + videos:
        url = item.get('url', '').lower()
        if 'chatgpt.com' in url:
            bots.add('ChatGPT')
        elif 'claude' in url:
            bots.add('Claude')
        elif 'gemini' in url:
            bots.add('Gemini')
    return sorted(list(bots))


def dedupe_by_url(items):
    seen = set()
    unique = []
    for item in items:
        if item['url'] not in seen:
            seen.add(item['url'])
            unique.append(item)
    return unique


def html_to_text(html, max_words=1000):
    parser = TextExtractor()
    parser.feed(html)
    text = re.sub(r'\s+', ' ', parser.get_text()).strip()
    return ' '.join(text.split()[:max_words])


def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


print("✅ Helper functions loaded")

## 📄 Extraction functions

In [ ]:
# A line that is just a month, like "June 2026", "December" or "May to June 2023"
MONTH_LINE = re.compile(r'(January|February|March|April|May|June|July|August|September|October|November|December)'
                        r'(\s+(to|-|–)\s+\w+)?(\s+\d{4})?', re.I)


def master_sections(master_html):
    """The master doc split at its headings and at lines that are just a month (months are newest first)."""
    cuts = [m.start() for m in re.finditer(r'<h[1-6][\s>]', master_html)]
    for m in re.finditer(r'<p[^>]*>(.*?)</p>', master_html, re.S):
        text = html_lib.unescape(re.sub(r'<[^>]+>', '', m.group(1))).replace('\xa0', ' ')
        if MONTH_LINE.fullmatch(re.sub(r'\s+', ' ', text).strip()):
            cuts.append(m.start())
    cuts = sorted(set([0] + cuts + [len(master_html)]))
    return [master_html[a:b] for a, b in zip(cuts, cuts[1:])]


def extract_post_urls_from_master(master_html):
    """The posts the master doc links to, newest first. Months are newest first in the doc,
    but the posts within each month are oldest first, so each month's posts are reversed."""
    posts, seen = [], set()
    for section in master_sections(master_html):
        parser = LinkExtractor()
        parser.feed(section)
        in_section = []
        for link in parser.links:
            url = unwrap_google_redirect(link['url'])
            if not is_artifact_link(link['text'], url) and is_google_doc_link(url):
                in_section.append({'url': url, 'title': link['text']})
        for post in reversed(in_section):
            key = doc_key(post['url'])
            if key not in seen:
                seen.add(key)
                posts.append(post)
    return posts


def extract_links_from_post(post_html, post_url, post_title):
    """Returns the post's apps, chat logs and videos, fetching each link's title."""
    apps, chats, videos = [], [], []
    seen = set()
    for link in extract_all_links(post_html):
        url = unwrap_google_redirect(link['url'])
        page = url.split('#')[0]   # links to sections of the same page count once
        if page in seen or is_artifact_link(link['text'], url) or is_google_doc_link(url):
            continue
        seen.add(page)
        link_type, default_title = classify_link(url, link['text'])
        before_count = len(broken_links)

        if link_type == 'app':
            apps.append({'url': url, 'title': fetch_page_title(url, default_title, link['text'])})
            time.sleep(0.2)
        elif link_type == 'chat':
            chats.append({'url': url, 'title': fetch_page_title(url, default_title, link['text'])})
            time.sleep(0.2)
        elif link_type == 'video':
            vid_id = extract_youtube_id(url)
            if vid_id:
                videos.append({'url': url, 'title': fetch_youtube_title(vid_id, link['text']), 'video_id': vid_id})
                time.sleep(0.2)

        # Tag any newly found broken links with the post they came from
        for i in range(before_count, len(broken_links)):
            broken_links[i]['post_title'] = post_title
            broken_links[i]['post_url'] = post_url

    return {'apps': apps, 'chat_logs': chats, 'videos': videos}


def generate_broken_links_report(broken_links, filename='broken_links_report.html'):
    if not broken_links:
        return None

    # Deduplicate by URL (keep first occurrence with its context)
    seen_urls = set()
    unique_links = []
    for link in broken_links:
        if link['url'] not in seen_urls:
            seen_urls.add(link['url'])
            unique_links.append(link)

    html_content = f'''<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Broken Links Report - Ken Kahn</title>
    {favicon_link('alert')}
    <style>
        * {{ margin: 0; padding: 0; box-sizing: border-box; }}
        body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Arial, sans-serif; background: #f5f7fa; padding: 2rem; }}
        .header {{ max-width: 1600px; margin: 0 auto 2rem; }}
        h1 {{ color: #e53935; font-size: 2rem; margin-bottom: 0.5rem; }}
        .summary {{ background: #fff3cd; border-left: 4px solid #ffc107; padding: 1rem; border-radius: 4px; margin-bottom: 2rem; }}
        .container {{ max-width: 1600px; margin: 0 auto; background: white; border-radius: 8px; box-shadow: 0 2px 8px rgba(0,0,0,0.1); overflow: hidden; }}
        table {{ width: 100%; border-collapse: collapse; }}
        thead {{ background: #f8f9fa; border-bottom: 2px solid #dee2e6; }}
        th {{ padding: 1rem; text-align: left; font-weight: 600; color: #495057; font-size: 0.85rem; text-transform: uppercase; letter-spacing: 0.5px; }}
        td {{ padding: 1rem; border-bottom: 1px solid #f1f3f5; vertical-align: top; }}
        tbody tr:hover {{ background: #f8f9fa; }}
        .error-badge {{ display: inline-block; padding: 0.25rem 0.6rem; border-radius: 12px; font-size: 0.75rem; font-weight: 600; background: #e53935; color: white; }}
        .type-badge {{ display: inline-block; padding: 0.25rem 0.6rem; border-radius: 12px; font-size: 0.75rem; font-weight: 600; margin-right: 0.5rem; }}
        .type-page {{ background: #ff8c00; color: white; }}
        .type-video {{ background: #e53935; color: white; }}
        .broken-url {{ font-family: monospace; font-size: 0.85rem; color: #495057; word-break: break-all; }}
        .link-context {{ font-style: italic; color: #666; font-size: 0.9rem; line-height: 1.5; }}
        .post-link {{ color: #667eea; text-decoration: none; }}
        .post-link:hover {{ text-decoration: underline; }}
    </style>
</head>
<body>
    <div class="header">
        <h1>⚠️ Broken Links Report</h1>
        <p style="color: #666; margin-top: 0.5rem;">Links in newly added posts that could not be accessed</p>
    </div>
    <div class="summary">
        <strong>📄 Summary:</strong> Found {len(unique_links)} unique broken link(s) during extraction.
        These links returned errors when attempting to fetch their titles or verify their existence.
    </div>
    <div class="container">
        <table>
            <thead>
                <tr>
                    <th style="width: 8%;">Type</th>
                    <th style="width: 30%;">Broken URL</th>
                    <th style="width: 25%;">Link Text / Context</th>
                    <th style="width: 12%;">Error</th>
                    <th style="width: 25%;">Found in Post</th>
                </tr>
            </thead>
            <tbody>
'''

    for link in unique_links:
        type_class = 'type-page' if link['type'] == 'page' else 'type-video'
        type_label = '🔧 App/Chat' if link['type'] == 'page' else '▶ Video'
        post_title = link.get('post_title', 'Unknown Post')
        post_url = link.get('post_url', '#')
        context = link.get('context', 'No context available')

        html_content += f'''                <tr>
                    <td><span class="type-badge {type_class}">{type_label}</span></td>
                    <td><div class="broken-url">{link['url']}</div></td>
                    <td><div class="link-context">"{context}"</div></td>
                    <td><span class="error-badge">{link['error']}</span></td>
                    <td>
                        <a href="{post_url}" target="_blank" class="post-link">{post_title[:60]}</a>
                    </td>
                </tr>
'''

    html_content += '''            </tbody>
        </table>
    </div>
    <div style="max-width: 1600px; margin: 2rem auto; padding: 1rem; background: #e3f2fd; border-radius: 4px; border-left: 4px solid #2196f3;">
        <strong>💡 Note:</strong> Links may be broken due to:
        <ul style="margin-top: 0.5rem; margin-left: 1.5rem; color: #666;">
            <li>Pages that have been deleted or moved</li>
            <li>Private/authentication-required content (403 errors are excluded)</li>
            <li>Temporary server issues or timeouts</li>
            <li>Invalid or malformed URLs</li>
        </ul>
    </div>
</body>
</html>'''

    with open(filename, 'w', encoding='utf-8') as f:
        f.write(html_content)
    print(f"⚠️  Broken links report: {filename}")
    return filename


print("✅ Extraction functions loaded")

## 🤖 AI descriptions and verdicts (OpenAI by default, or Gemini)

In [ ]:
_api_keys = {}

def get_api_key():
    """Colab secret, then environment variable, then ask (without echoing). Asked at most once."""
    name = 'OPENAI_API_KEY' if PROVIDER == 'openai' else 'GEMINI_API_KEY'
    if name in _api_keys:
        return _api_keys[name]
    key = ''
    try:
        from google.colab import userdata
        key = userdata.get(name) or ''
    except Exception:
        pass
    key = key or os.environ.get(name, '')
    if not key:
        from getpass import getpass
        where = 'https://platform.openai.com/api-keys' if PROVIDER == 'openai' else 'https://aistudio.google.com/app/apikey'
        key = getpass(f"Enter your {name} ({where}): ")
    _api_keys[name] = key.strip()
    return _api_keys[name]


def description_prompt(post, doc_preview):
    app_list = ', '.join([app['title'] for app in post.get('apps', [])[:5]]) or 'None'
    chat_list = ', '.join([chat['title'] for chat in post.get('chat_logs', [])[:5]]) or 'None'
    return f"""Write ONE engaging paragraph (3-5 sentences, max 150 words) describing this AI education research post.

Title: {post['title']}
Apps created: {app_list}
Chat logs: {chat_list}

Document preview:
{doc_preview[:2000]}

Focus on what was created or explored. Be specific and interesting."""


def call_openai(prompt, json_mode=False, max_tokens=4000):
    body = {"model": OPENAI_MODEL,
            "messages": [{"role": "user", "content": prompt}],
            "max_completion_tokens": max_tokens}   # generous: reasoning models spend some of this thinking
    if json_mode:
        body["response_format"] = {"type": "json_object"}
    req = urllib.request.Request("https://api.openai.com/v1/chat/completions",
                                 data=json.dumps(body).encode(),
                                 headers={'Content-Type': 'application/json',
                                          'Authorization': f'Bearer {get_api_key()}'})
    with urllib.request.urlopen(req, timeout=120) as resp:
        result = json.loads(resp.read().decode())
    return (result['choices'][0]['message'].get('content') or '').strip()


def call_gemini(prompt, json_mode=False, max_tokens=4000):
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{GEMINI_MODEL}:generateContent"
    config = {"temperature": 0.7, "maxOutputTokens": max_tokens}
    if json_mode:
        config["responseMimeType"] = "application/json"
    body = {"contents": [{"parts": [{"text": prompt}]}], "generationConfig": config}
    req = urllib.request.Request(url, data=json.dumps(body).encode(),
                                 headers={'Content-Type': 'application/json', 'x-goog-api-key': get_api_key()})
    with urllib.request.urlopen(req, timeout=120) as resp:
        result = json.loads(resp.read().decode())
    parts = result.get('candidates', [{}])[0].get('content', {}).get('parts') or []
    return (parts[0].get('text') or '').strip() if parts else ''


def ask_model(prompt, json_mode=False, max_retries=6, max_tokens=4000):
    """The model's reply, or None if it failed (the post is then retried on the next run)."""
    call = call_openai if PROVIDER == 'openai' else call_gemini
    for attempt in range(1, max_retries + 1):
        try:
            return call(prompt, json_mode, max_tokens) or None
        except urllib.error.HTTPError as e:
            if e.code in (429, 500, 502, 503, 504) and attempt < max_retries:
                wait = int(e.headers.get('Retry-After') or 0) or min(2 ** attempt, 60)
                print(f"   ⚠️  HTTP {e.code} — retrying in {wait}s ({attempt}/{max_retries})")
                time.sleep(wait)
                continue
            detail = e.read().decode(errors='ignore')[:300]
            print(f"   ❌ HTTP {e.code}: {detail}")
            if e.code in (401, 403):
                raise   # bad key: no point carrying on
            return None
        except Exception as e:
            print(f"   ❌ {e}")
            return None
    return None


def generate_description(post, doc_preview):
    return ask_model(description_prompt(post, doc_preview))


def verdict_prompt(post, doc_preview, examples):
    apps = '\n'.join(f"- {a['title']} ({a['url']})" for a in post.get('apps', [])[:12]) or 'None'
    on_list = '\n'.join(f"- {t}" for t, on in examples if on)
    off_list = '\n'.join(f"- {t}" for t, on in examples if not on)
    return f"""Ken Kahn writes posts about his research and keeps a second, shorter list of "posts about web
apps that use AI". Almost everything he makes is built together with AI chatbots, so being made WITH an AI's
help is not enough on its own. The list is for posts where a web app that uses AI itself — calling a language
model, image/speech generator or other machine-learning model while someone is using it — is a central part
of the post.

Here are his past decisions, which show where he draws the line.

ON the list:
{on_list}

NOT on the list:
{off_list}

Decide for this new post, judging the way Ken has.

Title: {post['title']}
Apps linked from the post:
{apps}

Document preview:
{doc_preview[:6000]}

Reply with JSON only: {{"uses_ai": true or false, "reason": "one short sentence"}}"""


def judge_ai_app(post, doc_preview, examples):
    """{'ai_app': bool, 'reason': str}, or None if it failed (the post is then retried on the next run)."""
    for _ in range(2):
        text = ask_model(verdict_prompt(post, doc_preview, examples), json_mode=True)
        if not text:
            return None
        try:
            result = json.loads(text[text.find('{'):text.rfind('}') + 1])
            return {'ai_app': bool(result['uses_ai']), 'reason': str(result.get('reason', '')).strip()}
        except Exception:
            print("   ⚠️  Unreadable verdict — asking again")
    return None


def link_texts(post_html):
    """What each link says in the post, e.g. 'the Claude artifact version'."""
    texts = {}
    for link in extract_all_links(post_html):
        url = unwrap_google_redirect(link['url'])
        if link['text'] and not texts.get(url):
            texts[url] = link['text']
    return texts


def sort_prompt(post, texts, doc_preview, known_names):
    links = '\n'.join(f"- {a['url']}\n    page title: {a['title']}\n    link text in the post: {texts.get(a['url']) or '(none)'}"
                      for a in post['apps'])
    known = '\n'.join(f"- {n}" for n in known_names) or '(none yet)'
    return f"""This is one of Ken Kahn's research posts. Below are the links in it that might be apps. Sort them.

1. Group links that are the SAME app. That includes copies of an app hosted in different places (a Claude
   artifact is usually a copy of an app that is also on GitHub Pages; the link text often says "the Claude
   artifact version"), and successive or alternative versions of one app (v1 and v2, with or without API keys,
   a translation, a version with added features). Different apps made in the same post are separate groups,
   and so are apps that different chatbots each built from the same idea (for example ChatGPT's, Claude's and
   Gemini's versions of an app, made to compare them).
2. Links that are not apps go under "docs": documentation, manuals, guides, "making of" stories, reports,
   transcripts, pages of notes.
3. Give each app a short proper name, as Ken would call it, never "Claude Artifact" or a file name. If it is one
   of the apps already known from earlier posts (listed below), use exactly that name.

Every link must appear exactly once: in one app group or under docs.

Post title: {post['title']}

Links:
{links}

Apps already known from earlier posts:
{known}

Post text:
{doc_preview[:8000]}

Reply with JSON only:
{{"apps": [{{"name": "...", "links": ["url", ...], "note": "a few words on how the links relate, e.g. 'artifact and GitHub Pages copies'"}}],
  "docs": ["url", ...]}}"""


def sort_app_links(post, post_html, known_names):
    """Splits a post's app links into apps (each tagged with its app's name) and docs. False if it failed."""
    if not post['apps']:
        post['docs'], post['links_sorted'] = post.get('docs', []), True
        return True
    same = lambda u: urllib.parse.unquote(u).strip()   # the model sometimes decodes %20 and the like
    originals = {same(a['url']): a['url'] for a in post['apps']}
    prompt = sort_prompt(post, link_texts(post_html), html_to_text(post_html, 1500), known_names)
    for _ in range(2):
        reply = ask_model(prompt, json_mode=True, max_tokens=16000)   # posts with many links need room to think
        if not reply:
            return False
        try:
            result = json.loads(reply[reply.find('{'):reply.rfind('}') + 1])
            placed = [same(u) for g in result['apps'] for u in g['links']] + [same(u) for u in result.get('docs', [])]
            placed = [u for u in placed if u in originals]   # ignore extra links it picked up from the post's text
            if sorted(placed) == sorted(originals):
                break
        except Exception:
            pass
        print("   ⚠️  The sorting didn't place every link exactly once — asking again")
    else:
        return False
    group_of = {originals[same(u)]: g for g in result['apps'] for u in g['links'] if same(u) in originals}
    apps, docs = [], list(post.get('docs', []))
    for a in post['apps']:
        if a['url'] in group_of:
            g = group_of[a['url']]
            apps.append({**a, 'app': str(g['name']).strip(), 'app_note': str(g.get('note', '')).strip()})
        else:
            docs.append({'url': a['url'], 'title': a['title']})
    post['apps'], post['docs'], post['links_sorted'] = apps, docs, True
    return True


def consolidate_app_names(posts):
    """Asks the model which app names, across all posts, are the same app. Returns {name: name to use}."""
    seen = {}   # in the order the names first appear; posts are listed newest first
    for p in posts:
        for a in p.get('apps', []):
            if 'app' in a:
                info = seen.setdefault(a['app'], {'links': [], 'posts': []})
                if a['url'] not in info['links']:
                    info['links'].append(a['url'])
                if p['title'] not in info['posts']:
                    info['posts'].append(p['title'])
    lines = '\n'.join(f"- {name}  | e.g. {info['links'][0]}  | in: {'; '.join(t[:70] for t in info['posts'][:2])}"
                      for name, info in seen.items())
    prompt = f"""Below are the names given to apps in Ken Kahn's research posts, newest first, each with one of its
links and the post(s) it appears in. The names were chosen post by post, so a few different names may be the SAME
app: one app named differently in different posts, or successive versions of one app (often the same address,
or a later post continuing the same project). Merge only when it is clearly the same app, not just a similar
topic. Different apps must stay separate: for example two different games about unicorns, or apps that
different chatbots each built from the same idea (ChatGPT's, Claude's and Gemini's versions, made to compare them).

For each set of names that are the same app, give one clean name for it: the app's own name as its newest post
calls it, without version numbers, hosting details or other qualifiers in parentheses.

{lines}

Reply with JSON only, listing just the sets to merge:
{{"same": [{{"names": ["name as listed", "another name as listed", ...], "name": "clean name"}}, ...]}}"""
    reply = ask_model(prompt, json_mode=True, max_tokens=32000) or ''   # a long list to think about
    try:
        groups = json.loads(reply[reply.find('{'):reply.rfind('}') + 1]).get('same', [])
    except Exception:
        print("   ⚠️  Couldn't read the AI's list of app names to merge")
        return {}
    merges = {}
    for g in groups:
        names = [n for n in g.get('names', []) if n in seen]
        clean = str(g.get('name', '')).strip()
        if len(names) >= 2 and clean:
            merges.update({n: clean for n in names if n != clean})
    return merges


def has_good_description(post):
    d = (post.get('description') or '').strip()
    return bool(d) and not d.startswith('Research about ')


print(f"✅ AI functions loaded (provider: {PROVIDER}, model: {OPENAI_MODEL if PROVIDER == 'openai' else GEMINI_MODEL})")

## 🌐 HTML table

In [ ]:
FAVICONS = {
    # posts table: a little table on the page's purple
    'table': '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 64 64"><rect width="64" height="64" rx="14" fill="#667eea"/>'
             '<g fill="none" stroke="#fff" stroke-width="4" stroke-linecap="round"><rect x="13" y="15" width="38" height="34" rx="5"/>'
             '<path d="M13 27h38M13 38h38M27 27v22"/></g></svg>',
    # AI apps: sparkles on the app badges' orange
    'sparkle': '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 64 64"><rect width="64" height="64" rx="14" fill="#ff8c00"/>'
               '<path fill="#fff" d="M27 12c2 11 6 15 17 17-11 2-15 6-17 17-2-11-6-15-17-17 11-2 15-6 17-17z"/>'
               '<path fill="#fff" d="M47 36c1 6 3 8 9 9-6 1-8 3-9 9-1-6-3-8-9-9 6-1 8-3 9-9z"/></svg>',
    # broken links report
    'alert': '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 64 64"><rect width="64" height="64" rx="14" fill="#e53935"/>'
             '<rect x="28" y="13" width="8" height="26" rx="4" fill="#fff"/><circle cx="32" cy="49" r="5" fill="#fff"/></svg>',
}


def favicon_href(name):
    return 'data:image/svg+xml,' + urllib.parse.quote(FAVICONS[name])


def favicon_link(name):
    return f'<link rel="icon" id="favicon" type="image/svg+xml" href="{favicon_href(name)}">'


def generate_redirect(target, title):
    """A page for an old address that forwards to `target`."""
    return f'''<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>{title}</title>
    {favicon_link('sparkle')}
    <meta http-equiv="refresh" content="0; url={target}">
    <script>location.replace({json.dumps(target)});</script>
</head>
<body>
    <p>This table is now part of <a href="{target}">the table of all research posts</a>.</p>
</body>
</html>
'''


def generate_html(posts, doc_title, ai_title):
    """The table. Posts with ai_app true are the ones shown in the AI-apps view."""
    links = lambda key: {x['url'] for p in posts for x in p.get(key, [])}
    total_posts = len(posts)
    unique_apps = len({a.get('app') or a['url'] for p in posts for a in p['apps']})
    app_versions = len(links('apps'))
    total_docs = len(links('docs'))
    total_chats = len(links('chat_logs'))
    total_videos = len(links('videos'))
    posts_json_str = json.dumps(posts).replace('</', '<\\/')

    return f'''<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{doc_title}</title>
    {favicon_link('table')}
    <script src="https://cdn.jsdelivr.net/npm/marked/marked.min.js"></script>
    <style>
        * {{ margin: 0; padding: 0; box-sizing: border-box; }}
        body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Arial, sans-serif; background: #f5f7fa; padding: 2rem; }}
        .view-toggle {{ display: flex; justify-content: center; flex-wrap: wrap; gap: 0.5rem; margin-bottom: 1.5rem; }}
        .view-toggle button {{ font: inherit; font-size: 0.95rem; font-weight: 600; padding: 0.55rem 1.25rem; border-radius: 999px; border: 2px solid #dee2e6; background: white; color: #495057; cursor: pointer; transition: all 0.2s; }}
        .view-toggle button:hover {{ border-color: #667eea; }}
        #showAll[aria-pressed="true"] {{ background: #667eea; border-color: #667eea; color: white; }}
        #showAI[aria-pressed="true"] {{ background: #ff8c00; border-color: #ff8c00; color: white; }}
        .stats {{ display: flex; justify-content: center; flex-wrap: wrap; gap: 1rem 3rem; margin-bottom: 2rem; }}
        .stat {{ text-align: center; position: relative; cursor: help; border-radius: 8px; padding: 0.25rem 0.5rem; }}
        .stat:focus-visible {{ outline: 2px solid #667eea; outline-offset: 2px; }}
        button.stat {{ font: inherit; background: none; border: none; cursor: pointer; }}
        button.stat .stat-number {{ text-decoration: underline dotted 3px; text-underline-offset: 8px; }}
        button.stat:hover .stat-number {{ color: #5568d3; }}
        /* explanations: shown on hover, or on tap / keyboard focus */
        .stat::after {{ content: attr(data-tip); position: absolute; top: 100%; left: 50%; transform: translateX(-50%); margin-top: 0.4rem; width: max-content; max-width: min(260px, 80vw); padding: 0.6rem 0.75rem; border-radius: 6px; background: #2d3748; color: white; font-size: 0.8rem; font-weight: 400; line-height: 1.4; text-align: left; text-transform: none; letter-spacing: normal; opacity: 0; visibility: hidden; transition: opacity 0.15s; z-index: 10; pointer-events: none; }}
        .stat:hover::after, .stat:focus::after {{ opacity: 1; visibility: visible; }}
        .stat-number {{ font-size: 3rem; font-weight: 700; color: #667eea; }}
        .stat-label {{ font-size: 0.9rem; color: #666; text-transform: uppercase; letter-spacing: 1px; }}
        .container {{ max-width: 1600px; margin: 0 auto; background: white; border-radius: 8px; box-shadow: 0 2px 8px rgba(0,0,0,0.1); overflow: hidden; }}
        table {{ width: 100%; border-collapse: collapse; }}
        thead {{ background: #f8f9fa; border-bottom: 2px solid #dee2e6; }}
        th {{ padding: 1rem; text-align: left; font-weight: 600; color: #495057; font-size: 0.85rem; text-transform: uppercase; letter-spacing: 0.5px; }}
        td {{ padding: 1rem; border-bottom: 1px solid #f1f3f5; vertical-align: top; }}
        tbody tr:hover {{ background: #f8f9fa; }}
        .post-title {{ font-weight: 600; color: #2d3748; }}
        .post-title a {{ color: inherit; text-decoration: none; }}
        .post-title a:hover {{ color: #667eea; }}
        .info-btn {{ display: inline-block; width: 20px; height: 20px; line-height: 18px; text-align: center; background: #667eea; color: white; border-radius: 50%; cursor: pointer; font-size: 12px; font-weight: bold; margin-left: 0.5rem; border: none; transition: all 0.2s; }}
        .info-btn:hover {{ background: #5568d3; transform: scale(1.1); }}
        .badge {{ display: inline-block; padding: 0.35rem 0.7rem; border-radius: 4px; font-size: 0.8rem; font-weight: 600; margin: 0.2rem; text-decoration: none; transition: all 0.2s; }}
        .chatbot-badge {{ padding: 0.25rem 0.6rem; border-radius: 12px; font-size: 0.75rem; margin-right: 0.5rem; margin-bottom: 0.25rem; }}
        .chatbot-chatgpt {{ background: #10a37f; color: white; }}
        .chatbot-claude {{ background: #d97757; color: white; }}
        .chatbot-gemini {{ background: #4285f4; color: white; }}
        .badge-app {{ background: #ff8c00; color: white; }}
        .badge-app:hover {{ background: #ff7700; }}
        .badge-chat {{ background: #78909c; color: white; }}
        .badge-chat:hover {{ background: #607d8b; }}
        .badge-video {{ background: #e53935; color: white; }}
        .badge-video:hover {{ background: #c62828; }}
        .badge-app::before {{ content: "🔧 "; }}
        .badge-chat::before {{ content: "💬 "; }}
        .badge-video::before {{ content: "▶ "; }}
        .badge-doc {{ background: #8d6e63; color: white; }}
        .badge-doc:hover {{ background: #795548; }}
        .badge-doc::before {{ content: "📄 "; }}
        .app-list .intro {{ font-size: 0.9rem; color: #666; margin-bottom: 0.5rem; }}
        .app-list h3 {{ font-size: 1.05rem; color: #2d3748; margin: 1.2rem 0 0.3rem; }}
        .app-list .app-posts {{ font-size: 0.85rem; color: #666; line-height: 1.5; margin-top: 0.3rem; }}
        .app-list .app-posts a {{ color: #667eea; }}
        .modal {{ display: none; position: fixed; z-index: 1000; left: 0; top: 0; width: 100%; height: 100%; overflow: auto; background-color: rgba(0,0,0,0.5); }}
        .modal-content {{ background-color: #fefefe; margin: 5% auto; padding: 2rem; border-radius: 8px; width: 80%; max-width: 700px; box-shadow: 0 4px 20px rgba(0,0,0,0.3); max-height: 80vh; overflow-y: auto; }}
        .modal-header {{ display: flex; justify-content: space-between; align-items: center; margin-bottom: 1.5rem; padding-bottom: 1rem; border-bottom: 2px solid #f1f3f5; }}
        .modal-title {{ font-size: 1.5rem; font-weight: 700; color: #2d3748; flex: 1; }}
        .close {{ color: #aaa; font-size: 28px; font-weight: bold; cursor: pointer; border: none; background: none; padding: 0; margin-left: 1rem; }}
        .close:hover {{ color: #000; }}
        .modal-body {{ line-height: 1.8; color: #495057; }}
        .modal-body p {{ margin-bottom: 1rem; }}
    </style>
</head>
<body>
    <div class="view-toggle" role="group" aria-label="Which posts to show">
        <button type="button" id="showAll" aria-pressed="true">All posts</button>
        <button type="button" id="showAI" aria-pressed="false">✨ Web apps that use AI</button>
    </div>
    <div class="stats">
        <div class="stat" tabindex="0" data-tip="Research posts listed in the table."><div class="stat-number" id="statPosts">{total_posts}</div><div class="stat-label">Total Posts</div></div>
        <button type="button" class="stat" id="uniqueAppsButton" data-tip="Different apps. Copies of an app hosted in different places (such as a Claude artifact and the same app on GitHub Pages) and its successive versions count as one. AI grouped them from what the posts say. Click to see the list."><div class="stat-number" id="statUnique">{unique_apps}</div><div class="stat-label">Unique Apps</div></button>
        <div class="stat" tabindex="0" data-tip="Every distinct link to an app: each version and each hosted copy counts once, even when several posts link to it."><div class="stat-number" id="statVersions">{app_versions}</div><div class="stat-label">App Versions</div></div>
        <div class="stat" tabindex="0" data-tip="Manuals, guides, making-of stories and reports about the apps (distinct links)."><div class="stat-number" id="statDocs">{total_docs}</div><div class="stat-label">Docs</div></div>
        <div class="stat" tabindex="0" data-tip="Shared conversations with chatbots (distinct links)."><div class="stat-number" id="statChats">{total_chats}</div><div class="stat-label">Chat Logs</div></div>
        <div class="stat" tabindex="0" data-tip="Videos on YouTube (distinct links)."><div class="stat-number" id="statVideos">{total_videos}</div><div class="stat-label">Videos</div></div>
    </div>
    <div class="container">
        <table>
            <thead>
                <tr>
                    <th style="width: 35%;">Title</th>
                    <th style="width: 15%;">Chatbots</th>
                    <th style="width: 50%;"><span style="color: #ff8c00; font-weight: 600;">Apps</span>, <span style="color: #8d6e63; font-weight: 600;">Docs</span>, <span style="color: #78909c; font-weight: 600;">Chat Logs</span> & <span style="color: #e53935; font-weight: 600;">Videos</span></th>
                </tr>
            </thead>
            <tbody id="postsTable"></tbody>
        </table>
    </div>
    <div id="modal" class="modal">
        <div class="modal-content">
            <div class="modal-header">
                <div class="modal-title" id="modalTitle"></div>
                <button class="close">&times;</button>
            </div>
            <div class="modal-body" id="modalBody"></div>
        </div>
    </div>
    <script>
        const postsData = {posts_json_str};
        const VIEWS = {{
            all: {{ title: {json.dumps(doc_title)}, icon: {json.dumps(favicon_href('table'))} }},
            ai: {{ title: {json.dumps(ai_title)}, icon: {json.dumps(favicon_href('sparkle'))} }},
        }};
        let aiOnly = false;

        function esc(text) {{
            return String(text).replace(/[&<>"]/g, c => ({{ '&': '&amp;', '<': '&lt;', '>': '&gt;', '"': '&quot;' }})[c]);
        }}

        function visibleRows() {{
            return postsData.map((post, index) => [post, index]).filter(([post]) => !aiOnly || post.ai_app);
        }}

        function deduplicateLinks(links) {{
            const seen = new Set();
            return links.filter(link => {{
                if (seen.has(link.url)) return false;
                seen.add(link.url);
                return true;
            }});
        }}

        function renderPosts() {{
            const tbody = document.getElementById('postsTable');
            const rows = visibleRows();
            tbody.innerHTML = rows.map(([post, index]) => {{
                const chatbotBadges = post.chatbots.map(bot =>
                    `<span class="chatbot-badge chatbot-${{bot.toLowerCase()}}">${{bot}}</span>`
                ).join('');

                const allLinks = deduplicateLinks([
                    ...post.apps.map(app => ({{ url: app.url, title: app.title, type: 'app', tip: app.app ? 'App: ' + app.app : '' }})),
                    ...(post.docs || []).map(doc => ({{ url: doc.url, title: doc.title, type: 'doc', tip: 'Doc' }})),
                    ...post.chat_logs.map(chat => ({{ url: chat.url, title: chat.title, type: 'chat' }})),
                    ...post.videos.map(video => ({{ url: video.url, title: video.title, type: 'video' }}))
                ]);

                const linksHtml = allLinks.length > 0
                    ? allLinks.map(link => `<a href="${{link.url}}" target="_blank" class="badge badge-${{link.type}}"${{link.tip ? ` title="${{esc(link.tip)}}"` : ''}}>${{link.title}}</a>`).join('')
                    : '<span style="color: #999; font-style: italic;">No links</span>';

                return `
                    <tr>
                        <td>
                            <div class="post-title">
                                <a href="${{post.post_url}}" target="_blank">${{post.title}}</a>
                                <button class="info-btn" onclick="showDescription(${{index}})">i</button>
                            </div>
                        </td>
                        <td>${{chatbotBadges || '<span style="color: #999;">None</span>'}}</td>
                        <td>${{linksHtml}}</td>
                    </tr>
                `;
            }}).join('');

            const links = key => new Set(rows.flatMap(([post]) => (post[key] || []).map(x => x.url)));
            document.getElementById('statPosts').textContent = rows.length;
            document.getElementById('statUnique').textContent = new Set(rows.flatMap(([post]) => post.apps.map(a => a.app || a.url))).size;
            document.getElementById('statVersions').textContent = links('apps').size;
            document.getElementById('statDocs').textContent = links('docs').size;
            document.getElementById('statChats').textContent = links('chat_logs').size;
            document.getElementById('statVideos').textContent = links('videos').size;
        }}

        // The AI-apps view has its own address (#ai-apps), so it can be linked to and survives a reload
        function setView(ai, updateAddress) {{
            aiOnly = ai;
            const view = ai ? VIEWS.ai : VIEWS.all;
            document.title = view.title;
            document.getElementById('favicon').href = view.icon;
            document.getElementById('showAll').setAttribute('aria-pressed', String(!ai));
            document.getElementById('showAI').setAttribute('aria-pressed', String(ai));
            renderPosts();
            if (updateAddress) {{
                try {{ history.replaceState(null, '', ai ? '#ai-apps' : location.pathname + location.search); }}
                catch (e) {{ /* some previews don't allow changing the address */ }}
            }}
        }}
        document.getElementById('showAll').onclick = () => setView(false, true);
        document.getElementById('showAI').onclick = () => setView(true, true);
        window.addEventListener('hashchange', () => setView(location.hash === '#ai-apps', false));

        const modal = document.getElementById('modal');
        const closeBtn = document.querySelector('.close');

        function showDescription(index) {{
            const post = postsData[index];
            document.getElementById('modalTitle').textContent = post.title;
            const description = post.description || 'No description available.';
            const htmlContent = marked.parse(description);
            document.getElementById('modalBody').className = 'modal-body';
            document.getElementById('modalBody').innerHTML = htmlContent;
            modal.style.display = 'block';
        }}

        // The unique apps in the current view, each with its versions and copies and the posts it appears in
        function showAppList() {{
            const apps = new Map();
            for (const [post] of visibleRows()) {{
                for (const a of post.apps) {{
                    const key = a.app || a.url;
                    if (!apps.has(key)) apps.set(key, {{ name: a.app || a.title, links: new Map(), posts: new Map() }});
                    apps.get(key).links.set(a.url, a.title);
                    apps.get(key).posts.set(post.post_url, post.title);
                }}
            }}
            const list = [...apps.values()].sort((x, y) => x.name.localeCompare(y.name, undefined, {{ sensitivity: 'base' }}));
            document.getElementById('modalTitle').textContent = `Unique apps (${{list.length}})`;
            const body = document.getElementById('modalBody');
            body.className = 'modal-body app-list';
            body.innerHTML = '<p class="intro">Each app with its versions and copies, and the posts it appears in. '
                + 'AI did the grouping from what the posts say, so it can be wrong.</p>'
                + list.map(app => `<h3>${{esc(app.name)}}</h3>`
                    + [...app.links].map(([url, title]) => `<a href="${{esc(url)}}" target="_blank" class="badge badge-app">${{esc(title)}}</a>`).join('')
                    + `<div class="app-posts">${{app.posts.size === 1 ? 'Post' : 'Posts'}}: `
                    + [...app.posts].map(([url, title]) => `<a href="${{esc(url)}}" target="_blank">${{esc(title)}}</a>`).join(' · ')
                    + '</div>').join('');
            modal.style.display = 'block';
        }}
        document.getElementById('uniqueAppsButton').onclick = showAppList;

        closeBtn.onclick = function() {{ modal.style.display = 'none'; }}
        window.onclick = function(event) {{ if (event.target == modal) modal.style.display = 'none'; }}

        setView(location.hash === '#ai-apps', false);
    </script>
</body>
</html>'''


print("✅ HTML function loaded")

## 🔁 Incremental update

In [ ]:
def read_text(filename):
    """The file from the current folder if it's there, otherwise from GitHub. (None, None) if neither."""
    if os.path.exists(filename):
        with open(filename, encoding='utf-8') as f:
            return f.read(), 'local'
    try:
        req = urllib.request.Request(GITHUB_RAW + urllib.parse.quote(filename), headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=30) as resp:
            return resp.read().decode('utf-8'), 'GitHub'
    except Exception:
        return None, None


def load_posts(filename):
    """Posts from a .json file, or from the data embedded in a generated .html table."""
    text, source = read_text(filename)
    if not text:
        return [], None
    try:
        if filename.endswith('.json'):
            return json.loads(text), source
        match = re.search(r'const postsData = (\[.*?\]);\s*\n', text, re.S)
        return (json.loads(match.group(1)) if match else []), source
    except Exception as e:
        print(f"   ⚠️  Couldn't read {filename} ({source}): {e}")
        return [], None


def matches(post, patterns):
    key = doc_key(post.get('post_url') or post.get('url', ''))
    return any(p and (p == key or p.lower() in post['title'].lower()) for p in patterns)


def on_ai_list(post):
    if matches(post, AI_APPS_REMOVE):
        return False
    if matches(post, AI_APPS_ADD):
        return True
    return bool(post.get('ai_app'))


def load_known_posts():
    """Everything previously learned about posts, keyed by doc ID. The .json wins; the .html fills any gaps."""
    known = {}
    for filename in (OUTPUT + '.json', OUTPUT + '.html'):
        posts, source = load_posts(filename)
        if posts:
            print(f"   📂 {filename} ({source}): {len(posts)} posts")
        for p in posts:
            known.setdefault(doc_key(p.get('post_url', '')), dict(p))
    return known


def app_name(link):
    return APP_RENAMES.get(link['app'], link['app'])


def table_links(post):
    """A post's apps and docs as the table shows them, after the overrides in the configuration."""
    apps, docs = [], []
    for a in post.get('apps', []):
        if any(f and f in a['url'] for f in FORCE_DOC_LINKS):
            docs.append({'url': a['url'], 'title': a['title']})
        else:
            apps.append({**a, 'app': app_name(a)} if 'app' in a else a)
    for d in post.get('docs', []):
        if any(f and f in d['url'] for f in FORCE_APP_LINKS):
            apps.append({**d, 'app': APP_RENAMES.get(d['title'], d['title'])})
        else:
            docs.append(d)
    return apps, docs


def wants_refresh(entry):
    return REDO_ALL or matches(entry, REFRESH)


def save_json(posts, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(posts, f, indent=2, ensure_ascii=False)


def update_all():
    global broken_links
    broken_links = []
    print("=" * 80)
    files = [OUTPUT + '.json', OUTPUT + '.html'] + ([AI_APPS_REDIRECT] if AI_APPS_REDIRECT else [])
    before = {f: read_text(f)[0] or '' for f in files}   # read now: the checkpoints below rewrite the .json
    known = load_known_posts()

    master_html = fetch_google_doc_html(MASTER_URL)
    parser = TitleExtractor()
    parser.feed(master_html)
    master_title = parser.title or 'Ken Kahn Posts'
    listed = extract_post_urls_from_master(master_html)
    print(f"   📄 \"{master_title}\" lists {len(listed)} posts; {len(known)} posts already known")
    print("=" * 80 + "\n")

    posts, verdicts = [], []
    counts = {'new': 0, 'descriptions': 0, 'verdicts': 0, 'sorted': 0, 'failed': 0}
    main_output = OUTPUT + '.json'

    for i, entry in enumerate(listed, 1):
        key = doc_key(entry['url'])
        old = known.get(key)
        needs_links = old is None or wants_refresh(entry)
        needs_description = needs_links or not has_good_description(old)
        needs_verdict = old is None or 'ai_app' not in old
        needs_sorting = needs_links or not old.get('links_sorted')

        if not (needs_links or needs_description or needs_verdict or needs_sorting):
            post = dict(old)
            if post.get('title') != entry['title']:
                print(f"[{i}/{len(listed)}] ✏️  Title changed: {entry['title'][:60]}")
                post['title'] = entry['title']
            posts.append(post)
            continue

        todo = 'new' if old is None else ', '.join(w for w, n in (('refresh', needs_links and old),
                                                                   ('description', needs_description and not needs_links),
                                                                   ('AI-apps verdict', needs_verdict),
                                                                   ('sorting app links', needs_sorting and not needs_links)) if n)
        print(f"[{i}/{len(listed)}] {entry['title'][:60]}... ({'🆕 ' if old is None else ''}{todo})")
        counts['new'] += old is None
        if DRY_RUN:
            posts.append(dict(old) if old else {'title': entry['title'], 'post_url': entry['url'],
                                                'apps': [], 'chat_logs': [], 'videos': [], 'chatbots': []})
            continue

        try:
            post_html = fetch_google_doc_html(entry['url'])
        except Exception as e:
            print(f"   ❌ Couldn't fetch the post: {e}")
            post_html = ''

        if needs_links and post_html:
            result = extract_links_from_post(post_html, entry['url'], entry['title'])
            post = {'title': entry['title'], 'post_url': entry['url'], **result,
                    'chatbots': infer_chatbots(result['apps'], result['chat_logs'], result['videos'], entry['title'])}
            for k in ('ai_app', 'ai_app_reason'):
                if old and k in old:
                    post[k] = old[k]
            status = [f"{len(post[k])} {n}" for k, n in (('apps', 'apps'), ('chat_logs', 'chats'), ('videos', 'videos')) if post[k]]
            print(f"   {'✅ ' + ' | '.join(status) if status else '⚠️  No links'}")
        else:
            post = dict(old) if old else {'title': entry['title'], 'post_url': entry['url'],
                                          'apps': [], 'chat_logs': [], 'videos': [], 'chatbots': []}
            post['title'] = entry['title']

        text = html_to_text(post_html) if post_html else ''
        if needs_description:
            description = generate_description(post, text) if text else None
            if description:
                post['description'] = description
                counts['descriptions'] += 1
                print("   ✅ Description written")
            else:
                post.pop('description', None)
                counts['failed'] += 1

        if needs_verdict:
            examples = [(p['title'], on_ai_list(p)) for k, p in known.items() if k != key and 'ai_app' in p]
            verdict = judge_ai_app(post, text, examples) if text else None
            if verdict:
                post['ai_app'], post['ai_app_reason'] = verdict['ai_app'], verdict['reason']
                counts['verdicts'] += 1
                verdicts.append(post)
                print(f"   {'✨ On' if verdict['ai_app'] else '➖ Not on'} the AI-apps list: {verdict['reason']}")
            else:
                post.pop('ai_app', None)
                counts['failed'] += 1

        if needs_sorting:
            names = sorted({app_name(a) for k, p in known.items() if k != key for a in p.get('apps', []) if 'app' in a})
            if post_html and sort_app_links(post, post_html, names):
                counts['sorted'] += 1
                print(f"   🗂️  {len({a['app'] for a in post['apps']})} app(s), {len(post['docs'])} doc(s): "
                      + ', '.join(dict.fromkeys(a['app'] for a in post['apps'])))
            else:
                post.pop('links_sorted', None)
                counts['failed'] += 1

        known[key] = post
        posts.append(post)
        # checkpoint, so an interrupted run resumes here
        save_json(posts + [known[doc_key(e['url'])] for e in listed[i:] if doc_key(e['url']) in known], main_output)

    previous = json.loads(before[OUTPUT + '.json']) if before[OUTPUT + '.json'] else []
    listed_keys = {doc_key(e['url']) for e in listed}
    removed = [p['title'] for p in previous if doc_key(p.get('post_url', '')) not in listed_keys]
    for t in removed:
        print(f"   🗑️  No longer in the master doc: {t[:60]}")

    # Duplicate links within a post used to be stored twice, inflating the counts at the top
    for p in posts:
        for k in ('apps', 'docs', 'chat_logs', 'videos'):
            if k in p:
                p[k] = dedupe_by_url(p[k])

    if CONSOLIDATE_APP_NAMES and not DRY_RUN:
        merges = consolidate_app_names(posts)
        for p in posts:
            for a in p.get('apps', []):
                if a.get('app') in merges:
                    a['app'] = merges[a['app']]
        print(f"\nMerged {len(merges)} app names that were the same app:")
        for old, new in sorted(merges.items()):
            print(f"   {old}  →  {new}")

    print(f"\nSummary: {len(posts)} posts — {counts['new']} new, {counts['descriptions']} descriptions written, "
          f"{counts['verdicts']} AI-apps verdicts, {counts['sorted']} posts' app links sorted, "
          f"{counts['failed']} failures (retried next run), {len(removed)} removed")
    if verdicts:
        print("\nNew AI-apps verdicts (overrule with AI_APPS_ADD / AI_APPS_REMOVE):")
        for p in verdicts:
            print(f"   {'✨ ON ' if p['ai_app'] else '➖ off'}  {p['title'][:70]}")
    print()

    if DRY_RUN:
        print("(dry run — nothing saved)")
        return []

    # The table shows the overrides; the .json keeps the AI's own verdicts
    table_rows = [{**p, 'ai_app': on_ai_list(p), **dict(zip(('apps', 'docs'), table_links(p)))} for p in posts]
    outputs = {OUTPUT + '.json': json.dumps(posts, indent=2, ensure_ascii=False),
               OUTPUT + '.html': generate_html(table_rows, TITLE or master_title, AI_APPS_TITLE)}
    if AI_APPS_REDIRECT:
        outputs[AI_APPS_REDIRECT] = generate_redirect(OUTPUT + '.html#ai-apps', AI_APPS_TITLE)
    saved = []
    for filename, text in outputs.items():
        if text != before[filename]:
            with open(filename, 'w', encoding='utf-8') as f:
                f.write(text)
            saved.append(filename)
    print(f"{len(posts)} posts, {sum(p['ai_app'] for p in table_rows)} of them web apps that use AI; "
          f"{len({a.get('app') or a['url'] for p in table_rows for a in p['apps']})} unique apps")
    print(f"💾 Saved {', '.join(saved)}" if saved else "✅ Already up to date")
    report = generate_broken_links_report(broken_links)
    if report:
        saved.append(report)
    return saved


print("✅ Update functions loaded")

## ▶️ Update everything

In [ ]:
saved = update_all()

if saved:
    print("\n🎉 Updated: " + ', '.join(saved))
    if in_colab():
        from google.colab import files
        for filename in saved:
            files.download(filename)
else:
    print("\n🎉 Everything was already up to date")